# Price a one-shout option with QuantLib and PyTorch

A shout opportunity lets the holder lock intrinsic value while keeping the original contract's remaining option value. This notebook uses QuantLib's original `FdBlackScholesShoutEngine`, including its shout value calculator, exercise schedule and escrowed treatment of dated cash dividends.

Build and install QuantLibTorch first. This engine uses CPU float64 and prices one contract per call. Its volatility input is a constant Black volatility, not a sampled local-volatility surface.

In [1]:
from datetime import date
import torch
from quantlibtorch import (
    shout_mesh, fd_shout, FdShoutEngine, EquityMarket,
    Exercise, VanillaOption, PricingLayer,
)
torch.set_num_threads(1)
start, mid, end = date(2025, 1, 1), date(2025, 7, 1), date(2026, 1, 1)
volatility = torch.tensor(.2, dtype=torch.float64, requires_grad=True)

## Fix the mesh before differentiating

`shout_mesh` builds the original QuantLib mesh from passive numerical reference values. Reuse it for training and finite-difference checks so derivatives describe the pricing equations on fixed geometry. With cash dividends, mesh coordinates represent escrow-adjusted log spot.

The small grid below is for a quick demonstration. Resolution and domain accuracy need separate checks for the contracts and parameter ranges you use.

In [2]:
mesh = shout_mesh(.2, exercise='american', time_steps=12, x_grid=31)
option = VanillaOption(100., end, exercise=Exercise.american())
layer = PricingLayer(option, EquityMarket(),
    FdShoutEngine(mesh=mesh, time_steps=12, mode='adjoint'))
price = layer(volatility)
vega = torch.autograd.grad(price, volatility)[0]
print('American shout call and vega:', price.item(), vega.item())

American shout call and vega: 11.215897541086484 52.04949954576492


## Spot, rates and dated yield curves

Volatility, spot, flat rate/yield and noninitial dated discount factors remain graph connected. The rate and yield enter both rollback and the time-dependent value of shouting; the original financial dependencies are retained.

Configuration is fixed and each call builds a fresh graph. Supply tensor market values through call overrides, rather than storing an old autograd graph in `EquityMarket`.

In [3]:
spot = torch.tensor(100., dtype=torch.float64, requires_grad=True)
rate = torch.tensor(.03, dtype=torch.float64, requires_grad=True)
value = layer(volatility, spot=spot, rate=rate)
delta, rate_sensitivity = torch.autograd.grad(value, (spot, rate))
print('Spot and rate sensitivities:', delta.item(), rate_sensitivity.item())
discounts = torch.tensor([.982, .95], dtype=torch.float64, requires_grad=True)
value = layer(volatility, discount_curve=((start, 1.), (mid, discounts[0]), (end, discounts[1])))
print('Dated discount sensitivities:', torch.autograd.grad(value, discounts)[0].tolist())

Spot and rate sensitivities: 0.7340333336020731 44.12751388412874


Dated discount sensitivities: [-17.096019231546467, -40.520578148593664]


## Restrict shouting to Bermudan dates

Exercise dates specify the opportunities to shout. American allows opportunities throughout the interval; Bermudan uses the listed dates; European permits maturity only and reduces to the vanilla payoff. Cash dividends use this engine's original escrowed model. They are not spot-jump dividends.

Numerical geometry, strike, cash amounts and dates are passive configuration.

In [4]:
cash = ((mid, 1.25),)
bermudan_mesh = shout_mesh(.2, option_type='put', exercise='bermudan',
    exercise_dates=(mid, end), cash_dividends=cash, time_steps=12, x_grid=31)
bermudan = PricingLayer(
    VanillaOption(100., end, option_type='put', exercise=Exercise.bermudan((mid, end))),
    EquityMarket(cash_dividends=cash),
    FdShoutEngine(mesh=bermudan_mesh, time_steps=12, damping_steps=2, mode='adjoint'))
value = bermudan(volatility)
print('Bermudan shout put:', value.item())
print('Vega:', torch.autograd.grad(value, volatility)[0].item())

Bermudan shout put: 8.356279861968643


Vega: 44.777821278523646


## Higher derivatives and independent checks

Full AD supports higher reverse derivatives on the selected numerical branches. Equation adjoints are first order. Both preserve the original financial engine, schemes and damping. Exercise decisions can introduce nonsmooth parameter boundaries.

Repository tests compare anchor prices with the independent stock QuantLib 1.40 engine. Fixed-mesh sensitivities are checked with centered differences and full/scalar/adjoint comparisons. The stock Python wrapper does not expose the shout calculator needed for an independent fixed-mesh gradient oracle; its automatic-mesh bumps include geometry changes and are a different derivative.

In [5]:
full = PricingLayer(option, EquityMarket(),
    FdShoutEngine(mesh=mesh, time_steps=12, mode='full'))
value = full(volatility, spot=spot)
delta = torch.autograd.grad(value, spot, create_graph=True)[0]
gamma = torch.autograd.grad(delta, spot)[0]
h = 1e-5
with torch.no_grad():
    plus = fd_shout(volatility+h, mesh=mesh, time_steps=12)
    minus = fd_shout(volatility-h, mesh=mesh, time_steps=12)
print('Spot gamma:', gamma.item())
print('AD vega and same-mesh bump:', vega.item(), ((plus-minus)/(2*h)).item())

Spot gamma: 0.024543861844315562
AD vega and same-mesh bump: 52.04949954576492 52.04949954809023
